### Bronze to Silver Fact Notebook

Transforms raw Bronze fact and transaction files into cleansed, enriched, and joined Silver fact tables.

**Source:** Bronze Lakehouse &nbsp;&nbsp;|&nbsp;&nbsp; **Target:** Silver Lakehouse

**Input:** Purchase Order Header, Purchase Order Items, Account Assignment, Goods Receipt, Invoice Items

**Transformations:** remove duplicates · handle NULLs · trim whitespace · convert data types · business rule enrichment (PO age, delivery performance, assignment description) · join and consolidate into a single fact dataset · referential integrity checks between Header and Items · load audit column

**Output:** SILVER_purchase_order_header, SILVER_purchase_order_items, SILVER_account_assignment, SILVER_goods_receipt, SILVER_invoice_items, silver_fact_procurement (consolidated)

**Import PySpark functions and types**

In [1]:
from pyspark.sql.functions import*
from pyspark.sql.types import*

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 3, Finished, Available, Finished, False)

**Read Bronze fact and transaction source files (PO Header, PO Items, Account Assignment, Goods Receipt, Invoice Items)**

In [2]:
df_PO_Header=spark.read.format("csv").option("header","True").load("Files/BRONZE/FULL_LOAD/purchase_order_header.csv")
df_PO_Items=spark.read.format("csv").option("header","True").load("Files/BRONZE/FULL_LOAD/purchase_order_items.csv")
df_PO_Acc_Assign=spark.read.format("csv").option("header","True").load("Files/BRONZE/FULL_LOAD/account_assignment.csv")
df_Goods_Receipt=spark.read.format("csv").option("header","True").load("Files/BRONZE/FULL_LOAD/goods_receipt.csv")
df_Invoice_Items=spark.read.format("csv").option("header","True").load("Files/BRONZE/FULL_LOAD/invoice_items.csv")

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 4, Finished, Available, Finished, False)

**Preview all Bronze fact DataFrames**

In [3]:
display(df_PO_Header)
display(df_PO_Items)
display(df_PO_Acc_Assign)
display(df_Goods_Receipt)
display(df_Invoice_Items)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b2b6c57f-c2c8-43a3-bd9f-769efb7dbbad)

SynapseWidget(Synapse.DataFrame, d6260f3e-b8b2-406d-88ae-2db3636ef08a)

SynapseWidget(Synapse.DataFrame, 057564e4-1c7f-40ad-aa17-e8d83b68f14f)

SynapseWidget(Synapse.DataFrame, a75ec227-1ff0-490d-8ef0-6e0dc97da05d)

SynapseWidget(Synapse.DataFrame, b6ed95d9-a54a-44ab-a3fb-91d222790074)

**Define duplicate removal function**

In [4]:
# User def Dropping Dulicates

def drop_duplicates(df, keys, table_name):
    before = df.count()
    df_clean = df.dropDuplicates(keys)
    after = df_clean.count()
    removed = before - after
    print("=" * 60)
    print(f"DUPLICATE ANALYSIS : {table_name}")
    print("=" * 60)
    print(f"{'Before':<15} | {before}")
    print(f"{'After':<15} | {after}")
    print(f"{'Removed':<15} | {removed}")
    print("=" * 60)
    return df_clean

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 6, Finished, Available, Finished, False)

**Remove duplicate records across all fact tables**

In [5]:
# Dropping Dulicates

# ── Fact Tables ───────────────────────────────────────────
df_PO_Header     = drop_duplicates(df_PO_Header,["PO_Number"],"PO_Header")
df_PO_Items      = drop_duplicates(df_PO_Items,["PO_Number","PO_Line_Item"],"PO_Items")
df_PO_Acc_Assign = drop_duplicates(df_PO_Acc_Assign,["PO_Number","PO_Line_Item"],"Acc_Assignment")
df_Goods_Receipt = drop_duplicates(df_Goods_Receipt, ["Material_Doc_Number"],"Goods_Receipt")
df_Invoice_Items = drop_duplicates(df_Invoice_Items, ["Invoice_Doc_Number"],"Invoice_Items")


StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 7, Finished, Available, Finished, False)

DUPLICATE ANALYSIS : PO_Header
Before          | 302
After           | 302
Removed         | 0
DUPLICATE ANALYSIS : PO_Items
Before          | 339
After           | 339
Removed         | 0
DUPLICATE ANALYSIS : Acc_Assignment
Before          | 453
After           | 339
Removed         | 114
DUPLICATE ANALYSIS : Goods_Receipt
Before          | 339
After           | 339
Removed         | 0
DUPLICATE ANALYSIS : Invoice_Items
Before          | 339
After           | 339
Removed         | 0


**Define NULL value audit function**

In [6]:
# Finding nulls_User defined

def find_nulls(df, table_name, key_columns):

    print("=" * 100)
    print(f"TABLE : {table_name}")
    print("=" * 100)

    for c in df.columns:

        null_count = df.filter(col(c).isNull()).count()

        if null_count > 0:

            print(f"\nColumn : {c} | Null Count : {null_count}")

            display(
                df.filter(col(c).isNull())
                  .select(*key_columns, c)
            )

    print()

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 8, Finished, Available, Finished, False)

**Run NULL value audit across all fact tables**

In [7]:
# Finding nulls

tables = {
    "PO_Header": (df_PO_Header, ["PO_Number"]),
    "PO_Items": (df_PO_Items, ["PO_Number","PO_Line_Item"]),
    "Acc_Assignment": (df_PO_Acc_Assign, ["PO_Number","PO_Line_Item"]),
    "Goods_Receipt": (df_Goods_Receipt, ["Material_Doc_Number"]),
    "Invoice_Items": (df_Invoice_Items, ["Invoice_Doc_Number"]),
}

for table_name, (df, keys) in tables.items():
    find_nulls(df, table_name, keys)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 9, Finished, Available, Finished, False)

TABLE : PO_Header

Column : Vendor_ID | Null Count : 1


SynapseWidget(Synapse.DataFrame, df6e978b-534c-4fda-b03c-6775c27afd6b)


TABLE : PO_Items

Column : Net_Price | Null Count : 1


SynapseWidget(Synapse.DataFrame, 70e6dc34-2914-4095-a84d-7e69507b2dc7)


TABLE : Acc_Assignment

Column : GL_Account | Null Count : 1


SynapseWidget(Synapse.DataFrame, 07577fdb-735f-4e46-810d-f491f17145c2)


TABLE : Goods_Receipt

Column : Plant_ID | Null Count : 1


SynapseWidget(Synapse.DataFrame, 9ed9bd00-0ff5-44b0-a48d-8c84b7d7fb40)


TABLE : Invoice_Items



**Fill missing values with default or unknown placeholders**

In [8]:
print("=" * 60)
print("FILLING WITH UNKNOWN / DEFAULT VALUES")
print("=" * 60)

# ── PO_Header ────────────────────────────────────
df_PO_Header = df_PO_Header.fillna({"Vendor_ID": "100127"})

# ── Account Assignment ────────────────────────────────────
df_PO_Acc_Assign = df_PO_Acc_Assign.withColumn("Cost_Center_ID",
when((col("Account_Assignment_Category") == "K") &(col("Cost_Center_ID").isNull()), "UNKNOWN")
.otherwise(col("Cost_Center_ID")))

df_PO_Acc_Assign = df_PO_Acc_Assign.fillna({"GL_Account":"000000"})

# ── Goods Receipt ─────────────────────────────────────────
df_Goods_Receipt = df_Goods_Receipt.fillna({"Plant_ID":"PL03"})

print("All UNKNOWN/default fills complete")


StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 10, Finished, Available, Finished, False)

FILLING WITH UNKNOWN / DEFAULT VALUES
All UNKNOWN/default fills complete


**Define and run whitespace audit across fact tables**

In [9]:
def find_whitespace(df, table_name):
    print("=" * 60)
    print(f"WHITESPACE AUDIT : {table_name}")
    
    found = False
    for c in df.columns:
        # Compare length before and after trim
        diff = df.filter(
            length(col(c)) != length(trim(col(c)))
        ).count()
        if diff > 0:
            found = True
            print(f"{c:<30} | Rows with spaces: {diff}")
    if not found:
        print("No whitespace issues found ✓")
    print("=" * 60)

# ── Run across all tables ─────────────────────────────────
find_whitespace(df_PO_Header,"PO_Header")
find_whitespace(df_PO_Items,"PO_Items")
find_whitespace(df_PO_Acc_Assign,"Acc_Assignment")
find_whitespace(df_Goods_Receipt,"Goods_Receipt")
find_whitespace(df_Invoice_Items,"Invoice_Items")

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 11, Finished, Available, Finished, False)

WHITESPACE AUDIT : PO_Header
Currency_Code                  | Rows with spaces: 1
WHITESPACE AUDIT : PO_Items
No whitespace issues found ✓
WHITESPACE AUDIT : Acc_Assignment
No whitespace issues found ✓
WHITESPACE AUDIT : Goods_Receipt
No whitespace issues found ✓
WHITESPACE AUDIT : Invoice_Items
No whitespace issues found ✓


**Define and apply whitespace trimming to all string columns**

In [10]:
def trim_whitespace(df, table_name):
    print(f"TRIM WHITESPACE : {table_name}")
    string_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType)
    ]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
    print(f"✓ Trimmed {len(string_cols)} string columns")
    print("=" * 60)
    return df

df_PO_Header     = trim_whitespace(df_PO_Header,     "PO_Header")
df_PO_Items      = trim_whitespace(df_PO_Items,      "PO_Items")
df_PO_Acc_Assign = trim_whitespace(df_PO_Acc_Assign, "Acc_Assignment")
df_Goods_Receipt = trim_whitespace(df_Goods_Receipt, "Goods_Receipt")
df_Invoice_Items = trim_whitespace(df_Invoice_Items, "Invoice_Items")

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 12, Finished, Available, Finished, False)

TRIM WHITESPACE : PO_Header
✓ Trimmed 12 string columns
TRIM WHITESPACE : PO_Items
✓ Trimmed 14 string columns
TRIM WHITESPACE : Acc_Assignment
✓ Trimmed 11 string columns
TRIM WHITESPACE : Goods_Receipt
✓ Trimmed 12 string columns
TRIM WHITESPACE : Invoice_Items
✓ Trimmed 11 string columns


**Define function to apply correct data types to each column**

In [11]:
## Defining DataTypes

def apply_data_types(df):

    string_cols = ["PO_Number", "Company_Code", "Vendor_ID", "Purchasing_Org","Purchasing_Group_ID",
"Currency_Code", "Material_ID", "Plant_ID","Storage_Location", "Unit_Of_Measure", "Cost_Center_ID", "GL_Account",
"Internal_Order_ID", "WBS_Element", "Material_Doc_Number","Invoice_Doc_Number", "PO_Document_Type","Account_Assignment_Category",
"PO_Status","Item_Status", "Invoice_Status"]

    int_cols = ["PO_Line_Item","Fiscal_Year","Material_Doc_Year","Movement_Type","Item_Category"]

    double_cols = ["Order_Quantity","Received_Quantity","Invoiced_Quantity"]

    decimal_cols = ["Net_Price","Net_Order_Value","Account_Assignment_Amount","Invoice_Amount"]

    date_cols = ["PO_Date","PO_Last_Changed_Date","Created_Date","Last_Modified_Date","PO_Line_Last_Changed_Date",
"Posting_Date"]

    for c in string_cols:
        if c in df.columns:df = df.withColumn(c, col(c).cast("string"))

    for c in int_cols:
        if c in df.columns:df = df.withColumn(c, col(c).cast("int"))

    for c in double_cols:
        if c in df.columns:df = df.withColumn(c, col(c).cast("double"))

    for c in decimal_cols:
        if c in df.columns:df = df.withColumn(c, col(c).cast(DecimalType(18, 2)))

    for c in date_cols:
        if c in df.columns:df = df.withColumn(c, col(c).cast("date"))

    return df

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 13, Finished, Available, Finished, False)

**Apply data types across all fact tables and print schemas**

In [12]:
## Applying DataTypes

df_PO_Header = apply_data_types(df_PO_Header)
df_PO_Items = apply_data_types(df_PO_Items)
df_PO_Acc_Assign = apply_data_types(df_PO_Acc_Assign)
df_Goods_Receipt = apply_data_types(df_Goods_Receipt)
df_Invoice_Items = apply_data_types(df_Invoice_Items)
df_PO_Header.printSchema()
df_PO_Items.printSchema()
df_PO_Acc_Assign.printSchema()
df_Goods_Receipt.printSchema()
df_Invoice_Items.printSchema()

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 14, Finished, Available, Finished, False)

root
 |-- PO_Number: string (nullable = true)
 |-- Company_Code: string (nullable = true)
 |-- PO_Document_Type: string (nullable = true)
 |-- PO_Date: date (nullable = true)
 |-- Vendor_ID: string (nullable = false)
 |-- Purchasing_Org: string (nullable = true)
 |-- Purchasing_Group_ID: string (nullable = true)
 |-- Currency_Code: string (nullable = true)
 |-- PO_Status: string (nullable = true)
 |-- PO_Last_Changed_Date: date (nullable = true)
 |-- Created_Date: date (nullable = true)
 |-- Last_Modified_Date: date (nullable = true)

root
 |-- PO_Number: string (nullable = true)
 |-- PO_Line_Item: integer (nullable = true)
 |-- Material_ID: string (nullable = true)
 |-- Plant_ID: string (nullable = true)
 |-- Storage_Location: string (nullable = true)
 |-- Item_Category: integer (nullable = true)
 |-- Order_Quantity: double (nullable = true)
 |-- Unit_Of_Measure: string (nullable = true)
 |-- Net_Price: decimal(18,2) (nullable = true)
 |-- Net_Order_Value: decimal(18,2) (nullable = tr

**Transform PO Header — standardize status, flag open POs, calculate PO age**

In [13]:
### PO_HEADER Transformations

### PO_STATUS==================================================================

df_PO_Header = df_PO_Header.withColumn("PO_Status",initcap(col("PO_Status")))

df_PO_Header = df_PO_Header.withColumn("Is_Open_PO",
when(col("PO_Status").isin("OPEN", "Approved"), "Yes").otherwise("No"))

df_PO_Header.select("PO_Status","Is_Open_PO").show()

### PO_AGE_DAYS ==================================================================

df_PO_Header = df_PO_Header.withColumn("PO_Age_Days",datediff(current_date(), col("PO_Date")))

# Extract month for monthly trend analysis in Power BI
df_PO_Header = df_PO_Header.withColumn("PO_Month",month(col("PO_Date")))

# Extract year for year on year comparison
df_PO_Header = df_PO_Header.withColumn("PO_Year",year(col("PO_Date")))

# Extract quarter for quarterly reporting
df_PO_Header = df_PO_Header.withColumn("PO_Quarter",quarter(col("PO_Date")))

df_PO_Header.select("PO_Date","PO_Month","PO_Year","PO_Quarter","PO_Age_Days").show()

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 15, Finished, Available, Finished, False)

+---------+----------+
|PO_Status|Is_Open_PO|
+---------+----------+
|Cancelled|        No|
|Cancelled|        No|
|   Closed|        No|
|   Closed|        No|
|   Closed|        No|
|     Open|        No|
|   Closed|        No|
|     Open|        No|
| Approved|       Yes|
|Cancelled|        No|
|Cancelled|        No|
|     Open|        No|
| Approved|       Yes|
|Cancelled|        No|
|     Open|        No|
| Approved|       Yes|
| Approved|       Yes|
|   Closed|        No|
|   Closed|        No|
| Approved|       Yes|
+---------+----------+
only showing top 20 rows

+----------+--------+-------+----------+-----------+
|   PO_Date|PO_Month|PO_Year|PO_Quarter|PO_Age_Days|
+----------+--------+-------+----------+-----------+
|2023-08-17|       8|   2023|         3|       1048|
|2023-03-07|       3|   2023|         1|       1211|
|2024-03-27|       3|   2024|         1|        825|
|2023-11-15|      11|   2023|         4|        958|
|2024-06-25|       6|   2024|         2|        735

**Transform PO Items — standardize units, describe item category, derive net price**

In [14]:
### PO_ITEMS Transformations


df_PO_Items = df_PO_Items.withColumn("Unit_Of_Measure",upper(trim(col("Unit_Of_Measure"))))

df_PO_Items = df_PO_Items.withColumn("Item_Category_Desc",when(col("Item_Category") == "0", "Standard Item")
.when(col("Item_Category") == "1", "Subcontracting").when(col("Item_Category") == "2", "Limit/Blanket")
.otherwise("Unknown"))

df_PO_Items = df_PO_Items.withColumn("Net_Price",when(col("Net_Price").isNull() &
col("Net_Order_Value").isNotNull() & col("Order_Quantity").isNotNull() & (col("Order_Quantity") != 0),
round(col("Net_Order_Value") / col("Order_Quantity"), 2)).otherwise(col("Net_Price")))

df_PO_Items = df_PO_Items.withColumn("Net_Order_Value",when(col("Net_Order_Value").cast("double") < 0,
abs(col("Net_Order_Value").cast("double"))).otherwise(col("Net_Order_Value").cast("double")))

df_PO_Items = df_PO_Items.withColumn("Order_Quantity",
when(col("Order_Quantity").isNull() &
col("Net_Price").isNotNull() &
(col("Net_Price") != 0),
round(col("Net_Order_Value") / col("Net_Price"), 0)).otherwise(col("Order_Quantity")))

df_PO_Items = df_PO_Items.withColumn("Price_Band",
when(col("Net_Price").cast("double") < 100,   "Low")
.when(col("Net_Price").cast("double") < 1000, "Medium")
.when(col("Net_Price").cast("double") < 5000, "High")
.otherwise("Premium"))

df_PO_Items = df_PO_Items.withColumn("Order_Value_Band",
when(col("Net_Order_Value").cast("double") < 10000,  "Low")
.when(col("Net_Order_Value").cast("double") < 100000,"Medium")
.otherwise("High"))

df_PO_Items = df_PO_Items.withColumn("Item_Status",initcap(col("Item_Status")))

df_PO_Items = df_PO_Items.withColumn("Is_Delivered",when(col("Item_Status") == "Delivered", "Yes").otherwise("No"))

df_PO_Items = df_PO_Items.withColumn("Is_Invoiced",when(col("Item_Status") == "Invoiced", "Yes").otherwise("No"))

df_PO_Items = df_PO_Items.withColumn("Is_Cancelled",when(col("Item_Status") == "Cancelled", "Yes").otherwise("No"))

print("PO Items - transformations complete")
df_PO_Items.printSchema()
display(df_PO_Items)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 16, Finished, Available, Finished, False)

PO Items - transformations complete
root
 |-- PO_Number: string (nullable = true)
 |-- PO_Line_Item: integer (nullable = true)
 |-- Material_ID: string (nullable = true)
 |-- Plant_ID: string (nullable = true)
 |-- Storage_Location: string (nullable = true)
 |-- Item_Category: integer (nullable = true)
 |-- Order_Quantity: double (nullable = true)
 |-- Unit_Of_Measure: string (nullable = true)
 |-- Net_Price: double (nullable = true)
 |-- Net_Order_Value: double (nullable = true)
 |-- Item_Status: string (nullable = true)
 |-- PO_Line_Last_Changed_Date: date (nullable = true)
 |-- Created_Date: date (nullable = true)
 |-- Last_Modified_Date: date (nullable = true)
 |-- Item_Category_Desc: string (nullable = false)
 |-- Price_Band: string (nullable = false)
 |-- Order_Value_Band: string (nullable = false)
 |-- Is_Delivered: string (nullable = false)
 |-- Is_Invoiced: string (nullable = false)
 |-- Is_Cancelled: string (nullable = false)



SynapseWidget(Synapse.DataFrame, a96ab246-1dcf-411f-a83c-7e9ee15d6f98)

**Transform Account Assignment — standardize currency, describe assignment category**

In [15]:
# Assignment Category Description

df_PO_Acc_Assign = df_PO_Acc_Assign.withColumn("Currency_Code", upper(col("Currency_Code")))
df_PO_Acc_Assign.select("Currency_Code").show()

df_PO_Acc_Assign = df_PO_Acc_Assign.withColumn("Assignment_Description",
when(col("Account_Assignment_Category") == "K", "Cost Centre")
.when(col("Account_Assignment_Category") == "P", "Project")
.when(col("Account_Assignment_Category") == "F", "Internal Order")
.otherwise("Unknown"))

from pyspark.sql.functions import when, lit, col

df_PO_Acc_Assign = df_PO_Acc_Assign.withColumn("Cost_Center_ID",
when(col("Account_Assignment_Category") == "K",col("Cost_Center_ID")).otherwise(lit(None))) \
.withColumn("WBS_Element",when(col("Account_Assignment_Category") == "P",col("WBS_Element")).otherwise(lit(None))) \
.withColumn("Internal_Order_ID",
when(col("Account_Assignment_Category") == "F",col("Internal_Order_ID")).otherwise(lit(None)))

# ── 7. Add Amount Band (identify which cost centres/projects have highest spend)

df_PO_Acc_Assign = df_PO_Acc_Assign.withColumn("Amount_Band",
when(col("Account_Assignment_Amount") < 1000,"Low")
.when(col("Account_Assignment_Amount") < 10000,"Medium")
.when(col("Account_Assignment_Amount") < 100000,"High")
.otherwise("Premium"))

print("Account Assignment transformations complete")
display(df_PO_Acc_Assign)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 17, Finished, Available, Finished, False)

+-------------+
|Currency_Code|
+-------------+
|          GBP|
|          EUR|
|          USD|
|          USD|
|          USD|
|          GBP|
|          GBP|
|          USD|
|          EUR|
|          EUR|
|          INR|
|          INR|
|          EUR|
|          USD|
|          EUR|
|          EUR|
|          EUR|
|          GBP|
|          GBP|
|          EUR|
+-------------+
only showing top 20 rows

Account Assignment transformations complete


SynapseWidget(Synapse.DataFrame, d79dc67b-6b12-4ded-b724-458c6e8e4a6b)

**Transform Goods Receipt — calculate days to receive, classify delivery performance**

In [16]:
#Goods_Receipt Transformations

#── Days_To_Receive ────────────────────────────────────
# Days from PO creation to goods received
# Key supplier lead time KPI
df_Goods_Receipt = df_Goods_Receipt.join(df_PO_Header.select("PO_Number", "PO_Date"),on="PO_Number", how="left")\
.withColumn("Days_To_Receive",datediff(col("Posting_Date"), col("PO_Date")))
df_Goods_Receipt = df_Goods_Receipt.drop("PO_Date")

# ── 3. Delivery_Performance ───────────────────────────────
# Categorise supplier delivery speed
# Fast = within 1 week
# Normal = within 30 days
# Slow = over 30 days
df_Goods_Receipt = df_Goods_Receipt.withColumn("Delivery_Performance",
when(col("Days_To_Receive") <= 7,  "Fast")
.when(col("Days_To_Receive") <= 30, "Normal")
.when(col("Days_To_Receive") >  30, "Slow")
.otherwise("Unknown"))

# ── 4. Pending_Quantity ───────────────────────────────────
# Outstanding quantity not yet received
# Pending = Ordered - Received
# Shows delivery shortfall per PO line
df_Goods_Receipt = df_Goods_Receipt.join(df_PO_Items.select("PO_Number", "PO_Line_Item", "Order_Quantity"),
on=["PO_Number", "PO_Line_Item"], how="left")\
.withColumn("Pending_Quantity",
col("Order_Quantity").cast("double") -coalesce(col("Received_Quantity").cast("double"), lit(0.0)))

df_Goods_Receipt = df_Goods_Receipt.drop("Order_Quantity")

print("Goods Receipt transformations done")
df_Goods_Receipt.select("PO_Number", "Posting_Date","Days_To_Receive", "Delivery_Performance",
"Received_Quantity","Pending_Quantity").show(5)

display(df_Goods_Receipt)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 18, Finished, Available, Finished, False)

Goods Receipt transformations done
+----------+------------+---------------+--------------------+-----------------+----------------+
| PO_Number|Posting_Date|Days_To_Receive|Delivery_Performance|Received_Quantity|Pending_Quantity|
+----------+------------+---------------+--------------------+-----------------+----------------+
|4500000001|  2023-09-03|             17|              Normal|            211.0|             0.0|
|4500000002|  2023-04-14|             38|                Slow|             42.0|            73.0|
|4500000003|  2024-04-14|             18|              Normal|              0.0|           115.0|
|4500000003|  2024-04-14|             18|              Normal|              0.0|           119.0|
|4500000004|  2023-12-17|             32|                Slow|            161.0|           222.0|
+----------+------------+---------------+--------------------+-----------------+----------------+
only showing top 5 rows



SynapseWidget(Synapse.DataFrame, e3f42c17-da0b-4b1c-b467-45014f741b0f)

**Correct negative invoice amounts and standardize invoice status**

In [17]:
# ── 1. Fix negative Invoice_Amount ────────────────────────
# 2 rows found with negative Invoice_Amount
# Negative invoice amount has no business meaning
# Taking absolute value to correct sign error
df_Invoice_Items = df_Invoice_Items.withColumn("Invoice_Amount",when(col("Invoice_Amount").cast("double") < 0,
abs(col("Invoice_Amount").cast("double"))).otherwise(col("Invoice_Amount").cast("double")))

df_Invoice_Items = df_Invoice_Items.withColumn("Invoice_Status",initcap(col("Invoice_Status")))

# Days from PO creation to invoice posting
df_Invoice_Items = df_Invoice_Items.join(df_PO_Header.select("PO_Number", "PO_Date"),on="PO_Number", how="left")\
.withColumn("Days_To_Invoice",datediff(col("Posting_Date"), col("PO_Date")))
df_Invoice_Items = df_Invoice_Items.drop("PO_Date")

#Is_Blocked flag ────────────────────────────────────
# Blocked invoices require manual intervention before payment
# High blocked % signals process or supplier data issues
# Important exception tracking KPI for finance team
df_Invoice_Items = df_Invoice_Items.withColumn("Is_Blocked",when(col("Invoice_Status") == "Blocked", "Yes")
.otherwise("No"))

print("Invoice Items transformations done")
df_Invoice_Items.select("PO_Number", "Invoice_Amount","Invoice_Status", "Days_To_Invoice","Is_Blocked").show(5)

display(df_Invoice_Items)


StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 19, Finished, Available, Finished, False)

Invoice Items transformations done
+----------+--------------+--------------+---------------+----------+
| PO_Number|Invoice_Amount|Invoice_Status|Days_To_Invoice|Is_Blocked|
+----------+--------------+--------------+---------------+----------+
|4500000001|     513719.59|        Posted|           1054|        No|
|4500000002|      197360.7|        Posted|           1217|        No|
|4500000003|     128589.55|       Blocked|            831|       Yes|
|4500000003|     340087.72|        Posted|            831|        No|
|4500000004|     1127406.5|        Posted|            964|        No|
+----------+--------------+--------------+---------------+----------+
only showing top 5 rows



SynapseWidget(Synapse.DataFrame, 4b6a4c58-d2fd-43ae-961c-94745f7036d6)

**Join PO Header with related tables to build an enriched PO dataset**

In [18]:
####  DATA ENRICHMENT (JOIN MULTIPLE SOURCES) 

# ── JOIN 1 — Start with PO Header ────────────────────────
df_PO_Full = df_PO_Header.alias("h").select(
col("h.PO_Number"),
col("h.Company_Code"),
col("h.PO_Document_Type"),
col("h.PO_Date"),
col("h.Vendor_ID"),
col("h.Purchasing_Org"),
col("h.Purchasing_Group_ID"),
col("h.Currency_Code").alias("PO_Currency_Code"),
col("h.PO_Status"),
col("h.PO_Last_Changed_Date"),
col("h.Created_Date").alias("PO_Created_Date"),
col("h.Last_Modified_Date").alias("PO_Last_Modified_Date"))

print(f"Step 1 — PO Header: {df_PO_Full.count()} rows | {len(df_PO_Full.columns)} cols")

# ── JOIN 2 — + PO Items ───────────────────────────────────
df_PO_Full = df_PO_Full.join(
    df_PO_Items.alias("i").select(
col("i.PO_Number"),
col("i.PO_Line_Item"),
col("i.Material_ID"),
col("i.Plant_ID"),
col("i.Storage_Location"),
col("i.Item_Category"),
col("i.Order_Quantity"),
col("i.Unit_Of_Measure"),
col("i.Net_Price"),
col("i.Net_Order_Value"),
col("i.Item_Status"),
col("i.PO_Line_Last_Changed_Date"),
col("i.Created_Date").alias("Item_Created_Date"),
col("i.Last_Modified_Date").alias("Item_Last_Modified_Date")),on="PO_Number",how="left")

print(f"Step 2 — + PO Items: {df_PO_Full.count()} rows | {len(df_PO_Full.columns)} cols")

# ── JOIN 3 — + Account Assignment ────────────────────────
df_PO_Full = df_PO_Full.join(df_PO_Acc_Assign.alias("a").select(
col("a.PO_Number"),
col("a.PO_Line_Item"),
col("a.Account_Assignment_Category"),
col("a.Cost_Center_ID"),
col("a.GL_Account"),
col("a.Internal_Order_ID"),
col("a.WBS_Element"),
col("a.Account_Assignment_Amount"),
col("a.Currency_Code").alias("AA_Currency_Code"),
col("a.Created_Date").alias("AA_Created_Date"),
col("a.Last_Modified_Date").alias("AA_Last_Modified_Date")),
on=["PO_Number", "PO_Line_Item"],how="left")

print(f"Step 3 — + Acc Assignment  : {df_PO_Full.count()} rows | {len(df_PO_Full.columns)} cols")

# ── JOIN 4 — + Goods Receipt ──────────────────────────────
df_PO_Full = df_PO_Full.join(df_Goods_Receipt.alias("g").select(
col("g.PO_Number"),
col("g.PO_Line_Item"),
col("g.Material_Doc_Number"),
col("g.Material_Doc_Year"),
col("g.Received_Quantity"),
col("g.Movement_Type"),
col("g.Posting_Date").alias("GR_Posting_Date"),
col("g.Created_Date").alias("GR_Created_Date"),
col("g.Last_Modified_Date").alias("GR_Last_Modified_Date")),on=["PO_Number", "PO_Line_Item"],how="left")

print(f"Step 4 — + Goods Receipt : {df_PO_Full.count()} rows | {len(df_PO_Full.columns)} cols")

# ── JOIN 5 — + Invoice Items ──────────────────────────────
df_PO_Full = df_PO_Full.join(
df_Invoice_Items.alias("inv").select(
col("inv.PO_Number"),
col("inv.PO_Line_Item"),
col("inv.Invoice_Doc_Number"),
col("inv.Fiscal_Year"),
col("inv.Invoiced_Quantity"),
col("inv.Invoice_Amount"),
col("inv.Invoice_Status"),
col("inv.Posting_Date").alias("Inv_Posting_Date"),
col("inv.Created_Date").alias("Inv_Created_Date"),
col("inv.Last_Modified_Date").alias("Inv_Last_Modified_Date")),on=["PO_Number", "PO_Line_Item"],how="left")

print(f"Step 5 — + Invoice Items   : {df_PO_Full.count()} rows | {len(df_PO_Full.columns)} cols")

# ── Final Summary ─────────────────────────────────────────
print("\n" + "=" * 50)
print(f"Final df_PO_Full")
print(f"Rows    : {df_PO_Full.count()}")
print(f"Columns : {len(df_PO_Full.columns)}")
print("=" * 50)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 20, Finished, Available, Finished, False)

Step 1 — PO Header: 302 rows | 12 cols
Step 2 — + PO Items: 339 rows | 25 cols
Step 3 — + Acc Assignment  : 339 rows | 34 cols
Step 4 — + Goods Receipt : 339 rows | 41 cols
Step 5 — + Invoice Items   : 339 rows | 49 cols

Final df_PO_Full
Rows    : 339
Columns : 49


**Preview the enriched, joined PO dataset**

In [19]:
display(df_PO_Full)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26255498-e3ec-4102-8c8b-ba0dd2cbf66a)

**Check for duplicate column names introduced by the joins**

In [20]:
# Check all duplicate columns first
cols = df_PO_Full.columns
duplicates = [c for c in cols if cols.count(c) > 1]
print("Duplicate columns:", set(duplicates))

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 22, Finished, Available, Finished, False)

Duplicate columns: set()


**Compare row counts between the joined dataset and PO Items**

In [21]:
print("PO_Full_count_after_joins : ",df_PO_Full.count())
print("PO_Line_Items_Count : ",df_PO_Items.count())

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 23, Finished, Available, Finished, False)

PO_Full_count_after_joins :  339
PO_Line_Items_Count :  339


**Identify PO numbers present in Header but missing from Items**

In [22]:
# Find PO_Numbers in Header but NOT in Items
df_header_only = df_PO_Header.join(
    df_PO_Items.select("PO_Number").distinct(),
    on="PO_Number",
    how="left_anti"  # rows in Header with no match in Items
)

print(f"PO_Numbers in Header but not in Items: {df_header_only.count()}")
df_header_only.select("PO_Number", "PO_Status", "Vendor_ID").show()

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 24, Finished, Available, Finished, False)

PO_Numbers in Header but not in Items: 0
+---------+---------+---------+
|PO_Number|PO_Status|Vendor_ID|
+---------+---------+---------+
+---------+---------+---------+



**Identify PO numbers present in Items but missing from Header**

In [23]:
# Find PO_Numbers in Items but NOT in Header
df_missing = df_PO_Items.join(
    df_PO_Header.select("PO_Number"),
    on="PO_Number",
    how="left_anti"   # ← rows in left NOT in right
)

print(f"PO_Items rows with no matching PO_Header : {df_missing.count()}")
df_missing.select("PO_Number").distinct().show()

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 25, Finished, Available, Finished, False)

PO_Items rows with no matching PO_Header : 0
+---------+
|PO_Number|
+---------+
+---------+



**Add Load_Date audit column to the consolidated fact dataset**

In [24]:
df_PO_Full = df_PO_Full.withColumn("Load_Date", current_timestamp())
display(df_PO_Full)

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a896837e-2307-4716-84a2-7a5edc7eae81)

**Write cleansed fact tables to Silver Lakehouse as Delta tables**

In [25]:
# Writing cleaned and transformed data into Delta format
# Each cleaned and transformed table saved separately
df_PO_Header.write.format("delta").mode("overwrite").saveAsTable("SILVER_purchase_order_header")
df_PO_Items.write.format("delta").mode("overwrite").saveAsTable("SILVER_purchase_order_items")
df_PO_Acc_Assign.write.format("delta").mode("overwrite").saveAsTable("SILVER_account_assignment")
df_Goods_Receipt.write.format("delta").mode("overwrite") .saveAsTable("SILVER_goods_receipt")
df_Invoice_Items.write.format("delta").mode("overwrite") .saveAsTable("SILVER_invoice_items")
# Joined result saved as one wide table
df_PO_Full.write.format("delta").mode("overwrite") .saveAsTable("SILVER_fact_procurement")

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 27, Finished, Available, Finished, False)

**Write cleansed fact tables to Silver Lakehouse as Parquet files**

In [26]:
df_PO_Header.write.mode("overwrite").parquet("Files/SILVER/FACT/silver_purchase_order_header")
df_PO_Items.write.mode("overwrite").parquet("Files/SILVER/FACT/silver_purchase_order_items")
df_PO_Acc_Assign.write.mode("overwrite").parquet("Files/SILVER/FACT/silver_account_assignment")
df_Goods_Receipt.write.mode("overwrite").parquet("Files/SILVER/FACT/silver_goods_receipt")
df_Invoice_Items.write.mode("overwrite").parquet("Files/SILVER/FACT/silver_invoice_items")
df_PO_Full.write.mode("overwrite").parquet("Files/SILVER/FACT/silver_fact_procurement")

StatementMeta(, da8f5fe5-caa2-4e13-846e-9a0eebe292e5, 28, Finished, Available, Finished, False)